#### Direction Confidence Model

The direction-confidence model predicts:

\[
P(\text{emitted direction is correct} \mid F(T))
\]

This is distinct from predicting whether the return is positive.

The selected direction model first emits a direction using the validation-selected
threshold of 0.64. A separate correctness model then estimates the probability that
this emitted direction will be realised.

To avoid evaluating confidence on the same observations used to fit it, the validation
period is divided chronologically into:

- Calibration block: fit the confidence model.
- Confidence-selection block: evaluate confidence quality.

The test split remains untouched.

Reconciliation note:

- The reproducible Notebook 05 confidence benchmark is `conf_direction_score=0.268893`.
- A higher score around `0.302190` appears only when the benchmark is rerun on the direction warm-up filtered validation sample (`53,909` rows) instead of the full Notebook 05 validation sample (`54,009` rows).
- The confidence features, logistic model, and scoring formula are otherwise unchanged.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss

SEED = 42
FINAL_DIRECTION_THRESHOLD = 0.64

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PANEL_PATH = (
    PROCESSED_DATA_DIR / "magnitude_model_panel.parquet"
)

DIRECTION_FEATURE_LIST_PATH = (
    PROCESSED_DATA_DIR / "final_direction_feature_columns.json"
)

print("Panel:", PANEL_PATH)
print("Feature list:", DIRECTION_FEATURE_LIST_PATH)

In [ ]:
if not PANEL_PATH.exists():
    panel_matches = list(
        PROJECT_ROOT.rglob("magnitude_model_panel.parquet")
    )

    assert len(panel_matches) == 1, (
        f"Expected one magnitude_model_panel.parquet, found: {panel_matches}"
    )

    PANEL_PATH = panel_matches[0]

if not DIRECTION_FEATURE_LIST_PATH.exists():
    feature_matches = list(
        PROJECT_ROOT.rglob(
            "final_direction_feature_columns.json"
        )
    )

    assert len(feature_matches) == 1, (
        f"Expected one feature-list file, found: {feature_matches}"
    )

    DIRECTION_FEATURE_LIST_PATH = feature_matches[0]

print("Resolved panel:", PANEL_PATH)
print("Resolved features:", DIRECTION_FEATURE_LIST_PATH)

In [ ]:
model_df = pd.read_parquet(
    PANEL_PATH
)

model_df["pred_date"] = pd.to_datetime(
    model_df["pred_date"]
)

model_df = (
    model_df
    .sort_values(["pred_date", "symbol"])
    .reset_index(drop=True)
)

with open(
    DIRECTION_FEATURE_LIST_PATH,
    "r",
    encoding="utf-8",
) as file:
    direction_feature_columns = json.load(file)

print("Shape:", model_df.shape)
print("Symbols:", model_df["symbol"].nunique())
print("Direction feature count:", len(direction_feature_columns))
print(model_df["split"].value_counts(dropna=False))

In [ ]:
required_columns = [
    "symbol",
    "pred_date",
    "split",
    "actual_direction",
    "actual_return_pct",
]

missing_required = [
    column
    for column in required_columns
    if column not in model_df.columns
]

missing_features = [
    column
    for column in direction_feature_columns
    if column not in model_df.columns
]

assert not missing_required, missing_required
assert not missing_features, missing_features

assert not model_df.duplicated(
    ["symbol", "pred_date"]
).any()

assert model_df["split"].isin(
    ["train", "valid", "test", "embargo"]
).all()

print("Direction-confidence input checks passed.")

In [ ]:
train_df = model_df[
    model_df["split"] == "train"
].copy()

valid_df = model_df[
    model_df["split"] == "valid"
].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Test rows untouched:", model_df["split"].eq("test").sum())

In [ ]:
X_train_direction = train_df[
    direction_feature_columns
].copy()

X_valid_direction = valid_df[
    direction_feature_columns
].copy()

y_train_direction = (
    train_df["actual_direction"]
    .eq(1)
    .astype("int8")
)

for frame in [
    X_train_direction,
    X_valid_direction,
]:
    frame["symbol"] = frame["symbol"].astype("category")

print("Train matrix:", X_train_direction.shape)
print("Validation matrix:", X_valid_direction.shape)

In [ ]:
direction_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1000,
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=100,
    feature_fraction=0.60,
    bagging_fraction=0.80,
    bagging_freq=1,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

direction_model.fit(
    X_train_direction,
    y_train_direction,
    categorical_feature=["symbol"],
)

print("Direction model refitted.")

In [ ]:
valid_probability_up = (
    direction_model.predict_proba(
        X_valid_direction
    )[:, 1]
)

valid_pred_direction = np.where(
    valid_probability_up >= FINAL_DIRECTION_THRESHOLD,
    1,
    -1,
)

valid_direction_correct = (
    valid_pred_direction
    == valid_df["actual_direction"].to_numpy()
).astype("int8")

valid_confidence_df = valid_df[
    [
        "symbol",
        "pred_date",
        "actual_direction",
        "actual_return_pct",
    ]
].copy()

valid_confidence_df["raw_probability_up"] = (
    valid_probability_up
)

valid_confidence_df["pred_direction"] = (
    valid_pred_direction
)

valid_confidence_df["direction_correct"] = (
    valid_direction_correct
)

print(
    valid_confidence_df[
        "direction_correct"
    ].value_counts(normalize=True)
)

In [ ]:
valid_confidence_df["probability_margin"] = np.abs(
    valid_confidence_df["raw_probability_up"]
    - FINAL_DIRECTION_THRESHOLD
)

valid_confidence_df["probability_distance_from_half"] = np.abs(
    valid_confidence_df["raw_probability_up"]
    - 0.50
)

valid_confidence_df["emitted_probability"] = np.where(
    valid_confidence_df["pred_direction"] == 1,
    valid_confidence_df["raw_probability_up"],
    1 - valid_confidence_df["raw_probability_up"],
)

CONFIDENCE_BASE_FEATURES = [
    "raw_probability_up",
    "probability_margin",
    "probability_distance_from_half",
    "emitted_probability",
    "pred_direction",
]

candidate_context_features = [
    "market_breadth",
    "aggregate_universe_volatility",
    "cross_sectional_return_dispersion",
    "day_of_week",
    "calendar_gap_days",
    "return_1d",
    "return_5d",
    "return_20d",
    "daily_volatility_20d",
    "overnight_volatility_20d",
    "volume_zscore_20d",
]

available_context_features = [
    column
    for column in candidate_context_features
    if column in valid_df.columns
]

for column in available_context_features:
    valid_confidence_df[column] = valid_df[column].to_numpy()

CONFIDENCE_FEATURE_COLUMNS = (
    CONFIDENCE_BASE_FEATURES
    + available_context_features
)

print("Confidence features:")
print(CONFIDENCE_FEATURE_COLUMNS)

In [ ]:
validation_dates = np.array(
    sorted(
        valid_confidence_df[
            "pred_date"
        ].drop_duplicates()
    )
)

calibration_cutoff_index = int(
    len(validation_dates) * 0.60
)

calibration_dates = set(
    validation_dates[
        :calibration_cutoff_index
    ]
)

confidence_selection_dates = set(
    validation_dates[
        calibration_cutoff_index:
    ]
)

confidence_calibration_df = (
    valid_confidence_df[
        valid_confidence_df["pred_date"].isin(
            calibration_dates
        )
    ]
    .copy()
)

confidence_selection_df = (
    valid_confidence_df[
        valid_confidence_df["pred_date"].isin(
            confidence_selection_dates
        )
    ]
    .copy()
)

print(
    "Calibration date range:",
    confidence_calibration_df["pred_date"].min(),
    "to",
    confidence_calibration_df["pred_date"].max(),
)

print(
    "Selection date range:",
    confidence_selection_df["pred_date"].min(),
    "to",
    confidence_selection_df["pred_date"].max(),
)

print("Calibration rows:", len(confidence_calibration_df))
print("Selection rows:", len(confidence_selection_df))

In [ ]:
X_conf_calibration = confidence_calibration_df[
    CONFIDENCE_FEATURE_COLUMNS
].copy()

y_conf_calibration = confidence_calibration_df[
    "direction_correct"
].copy()

X_conf_selection = confidence_selection_df[
    CONFIDENCE_FEATURE_COLUMNS
].copy()

y_conf_selection = confidence_selection_df[
    "direction_correct"
].copy()

numeric_fill_values = (
    X_conf_calibration
    .median(numeric_only=True)
)

X_conf_calibration = X_conf_calibration.fillna(
    numeric_fill_values
)

X_conf_selection = X_conf_selection.fillna(
    numeric_fill_values
)

print("Calibration matrix:", X_conf_calibration.shape)
print("Selection matrix:", X_conf_selection.shape)

In [ ]:
direction_confidence_model = LogisticRegression(
    C=0.25,
    max_iter=2000,
    class_weight=None,
    random_state=SEED,
)

direction_confidence_model.fit(
    X_conf_calibration,
    y_conf_calibration,
)

selection_conf_direction = (
    direction_confidence_model.predict_proba(
        X_conf_selection
    )[:, 1]
)

selection_conf_direction = np.clip(
    selection_conf_direction,
    0.500001,
    0.999999,
)

pd.Series(
    selection_conf_direction
).describe()

In [ ]:
shallow_confidence_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    num_leaves=7,
    learning_rate=0.03,
    min_child_samples=200,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=1,
    reg_lambda=2.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

shallow_confidence_model.fit(
    X_conf_calibration,
    y_conf_calibration,
)

selection_conf_direction_lgbm = (
    shallow_confidence_model.predict_proba(
        X_conf_selection
    )[:, 1]
)

selection_conf_direction_lgbm = np.clip(
    selection_conf_direction_lgbm,
    0.500001,
    0.999999,
)

pd.Series(
    selection_conf_direction_lgbm,
    name="shallow_lgbm_confidence",
).describe()

In [ ]:
def expected_calibration_error(
    actual_correct: np.ndarray,
    confidence: np.ndarray,
    n_bins: int = 10,
) -> float:
    actual_correct = np.asarray(
        actual_correct,
        dtype=float,
    )

    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    bin_edges = np.linspace(
        0,
        1,
        n_bins + 1,
    )

    bin_ids = np.digitize(
        confidence,
        bin_edges[1:-1],
        right=False,
    )

    ece = 0.0
    total_count = len(confidence)

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if not mask.any():
            continue

        bin_accuracy = actual_correct[mask].mean()
        bin_confidence = confidence[mask].mean()

        ece += (
            mask.sum() / total_count
        ) * abs(
            bin_accuracy - bin_confidence
        )

    return float(ece)


def confidence_direction_metrics(
    evaluation_df: pd.DataFrame,
    confidence: np.ndarray,
) -> dict:
    actual_return = evaluation_df[
        "actual_return_pct"
    ].to_numpy(dtype=float)

    predicted_direction = evaluation_df[
        "pred_direction"
    ].to_numpy(dtype=float)

    correct = evaluation_df[
        "direction_correct"
    ].to_numpy(dtype=float)

    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    weight = 2 * confidence - 1

    direction_score = (
        np.sum(
            predicted_direction
            * actual_return
        )
        / np.sum(
            np.abs(actual_return)
        )
    )

    conf_direction_score = (
        np.sum(
            weight
            * predicted_direction
            * actual_return
        )
        / np.sum(
            weight
            * np.abs(actual_return)
        )
    )

    brier = brier_score_loss(
        correct,
        confidence,
    )

    reference_probability = np.repeat(
        correct.mean(),
        len(correct),
    )

    brier_reference = brier_score_loss(
        correct,
        reference_probability,
    )

    brier_skill = (
        1 - brier / brier_reference
        if brier_reference > 0
        else np.nan
    )

    return {
        "direction_score": direction_score,
        "conf_direction_score": conf_direction_score,
        "conf_direction_lift": (
            conf_direction_score
            - direction_score
        ),
        "brier": brier,
        "brier_skill": brier_skill,
        "log_loss": log_loss(
            correct,
            np.clip(
                confidence,
                1e-6,
                1 - 1e-6,
            ),
        ),
        "ece_10": expected_calibration_error(
            correct,
            confidence,
            n_bins=10,
        ),
        "mean_confidence": confidence.mean(),
        "actual_accuracy": correct.mean(),
        "n_obs": len(correct),
    }

In [ ]:
raw_emitted_confidence = (
    confidence_selection_df[
        "emitted_probability"
    ].to_numpy()
)

raw_emitted_confidence = np.clip(
    raw_emitted_confidence,
    0.500001,
    0.999999,
)

raw_confidence_metrics = (
    confidence_direction_metrics(
        evaluation_df=confidence_selection_df,
        confidence=raw_emitted_confidence,
    )
)

correctness_model_metrics = (
    confidence_direction_metrics(
        evaluation_df=confidence_selection_df,
        confidence=selection_conf_direction,
    )
)

shallow_lgbm_confidence_metrics = (
    confidence_direction_metrics(
        evaluation_df=confidence_selection_df,
        confidence=selection_conf_direction_lgbm,
    )
)

direction_confidence_comparison = pd.DataFrame(
    [
        {
            "method": "raw_emitted_probability",
            **raw_confidence_metrics,
        },
        {
            "method": "correctness_logistic_model",
            **correctness_model_metrics,
        },
        {
            "method": "shallow_lgbm_correctness_model",
            **shallow_lgbm_confidence_metrics,
        },
    ]
)

direction_confidence_comparison

In [ ]:
print("Direction confidence notebook complete.\n")

print(pd.Series(correctness_model_metrics))

print("\nSelected model:")
print("Logistic Correctness Model")

print("\nConfidence Direction Score:",
      round(
          correctness_model_metrics[
              "conf_direction_score"
          ],
          6,
      ))

print("Confidence Lift:",
      round(
          correctness_model_metrics[
              "conf_direction_lift"
          ],
          6,
      ))

print("Next notebook:")
print("06_magnitude_confidence.ipynb")